# 02. Reinforcement Learning: Policy Gradient Methods

## Algorithm Category
**Type**: Reinforcement Learning - Policy-Based  
**Complexity**: High  
**Use Case**: Direct policy optimization using gradient ascent on expected return

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand policy gradient methods and REINFORCE algorithm
- Understand the difference between policy-based and value-based RL
- Implement REINFORCE from scratch
- Understand policy gradient theorem and why it works
- Compare policy-based vs value-based methods
- Visualize policy learning and reward improvement
- Apply policy gradients to simple environments
- Understand advantages and limitations of policy gradients

## Historical Context

Policy gradient methods were developed in the 1990s-2000s:
- Williams, R.J. (1992): "Simple statistical gradient-following algorithms"
- REINFORCE algorithm (1992): First practical policy gradient method
- Foundation for modern policy gradient methods (A3C, PPO, TRPO, etc.)
- Revolutionized continuous control and high-dimensional RL

**Key Papers/References:**
- Williams, R.J. (1992). "Simple statistical gradient-following algorithms for connectionist reinforcement learning"
- Sutton, R.S., et al. (2000). "Policy gradient methods for reinforcement learning"

## What are Policy Gradient Methods?

**Policy Gradient Methods** are a class of reinforcement learning algorithms that optimize the policy directly using gradient ascent. Instead of learning a value function (like Q-Learning), they learn the policy parameters that maximize expected return.

### Key Concepts

**Policy π(a|s)**: Probability distribution over actions given state
- Stochastic policy: outputs probabilities (not single action)
- Deterministic policy: outputs single action (special case)
- Parameterized: policy depends on parameters θ

**Policy Gradient**: Gradient of expected return with respect to policy parameters
- Points in direction of increasing expected return
- Used to update policy parameters via gradient ascent

**REINFORCE**: REward Increment = Nonnegative Factor × Offset Reinforcement × Characteristic Eligibility
- Monte Carlo policy gradient algorithm
- Uses complete episode returns
- High variance but unbiased

**Direct Policy Optimization**: Optimize policy directly (not via value function)
- No need to learn Q(s,a) or V(s)
- Can handle continuous action spaces naturally
- Better for high-dimensional problems

### Policy-Based vs Value-Based Methods

**Value-Based (Q-Learning)**:
- Learn value function Q(s,a) or V(s)
- Derive policy from value function
- Discrete actions work well
- Can be sample efficient

**Policy-Based (REINFORCE)**:
- Learn policy π(a|s) directly
- No value function needed
- Continuous actions work naturally
- Can learn stochastic policies

**Actor-Critic (Hybrid)**:
- Combines both approaches
- Actor: learns policy (policy-based)
- Critic: learns value function (value-based)
- Reduces variance of policy gradients

### When to Use Policy Gradient Methods

✅ **Good for:**
- Continuous action spaces (robotics, control)
- High-dimensional state spaces
- Stochastic policies needed
- Complex control problems
- When value function is hard to estimate
- When you need direct policy optimization
- Robotics and continuous control tasks

❌ **Not ideal for:**
- Simple discrete problems (Q-learning better)
- When sample efficiency is critical
- Very large action spaces
- When deterministic policy is sufficient
- Real-time applications (slow convergence)
- When value function is easy to learn

## Theory & Mechanics

### Mathematical Foundation

Policy gradient methods optimize the policy directly using gradient ascent.

**Policy Gradient Theorem:**
$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}[\nabla_\theta \log \pi_\theta(a|s) Q^{\pi_\theta}(s, a)]$$

**REINFORCE Algorithm:**
$$\nabla_\theta J(\theta) = \mathbb{E}[\nabla_\theta \log \pi_\theta(a|s) G_t]$$

Where $G_t$ is the return (sum of rewards from time t).

**Policy Update:**
$$\theta \leftarrow \theta + \alpha \nabla_\theta J(\theta)$$

**REINFORCE Update (Monte Carlo):**
$$\theta \leftarrow \theta + \alpha \gamma^t G_t \nabla_\theta \log \pi_\theta(a_t|s_t)$$

### How It Works

1. **Initialize**: Random policy parameters $\theta$
2. **Generate episode**: Follow current policy to collect trajectory
3. **Calculate returns**: Compute discounted returns $G_t$ for each step
4. **Update policy**: Gradient ascent on expected return
5. **Repeat**: Steps 2-4 until convergence

### Key Hyperparameters

- **learning_rate**: Step size for policy updates
- **gamma**: Discount factor for future rewards
- **baseline**: Variance reduction technique (optional)

### Advantages

- Works with continuous action spaces
- Can learn stochastic policies
- Direct policy optimization
- No need for value function
- Better for high-dimensional spaces

### Limitations

- High variance in gradient estimates
- Slow convergence
- Requires many samples
- May converge to local optima
- Sample inefficient


## Implementation

Let's implement REINFORCE algorithm from scratch.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Essential Tools for Policy Gradients
# ============================================

# NumPy: Numerical computing library
# Used for arrays, mathematical operations, random sampling, and probability calculations
import numpy as np

# Matplotlib: Plotting library
# Used for visualizing learning curves, policy probabilities, and performance metrics
import matplotlib.pyplot as plt

# Collections: Additional data structures
# deque: Double-ended queue (not used in this example, but useful for experience replay)
from collections import deque

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# SIMPLE ENVIRONMENT: Position Control Task
# ============================================

# SimpleEnv: Simple environment where agent tries to reach center position
# Similar to CartPole but simpler (1D position control)
class SimpleEnv:
    def __init__(self):
        """
        Initialize simple environment.
        Agent starts at position 0 and tries to stay at center.
        """
        self.state = 0  # Current position (state)
        self.max_steps = 20  # Maximum steps per episode
        
    def reset(self):
        """
        Reset environment to initial state.
        
        Returns:
            Initial state (0)
        """
        self.state = 0  # Reset to center position
        self.steps = 0  # Reset step counter
        return self.state  # Return initial state
    
    def step(self, action):
        """
        Execute action and return next state, reward, and done flag.
        
        Args:
            action: Action to take (0=left, 1=right)
        
        Returns:
            next_state: New position after action
            reward: Reward for this step
            done: Whether episode is finished
        """
        # Apply action: move left or right
        if action == 0:  # Left: decrease position
            self.state -= 1
        else:  # Right: increase position
            self.state += 1
        
        self.steps += 1  # Increment step counter
        
        # Reward: closer to center (0) is better
        # Negative absolute value: -|state|
        # At center (0): reward = 0 (best)
        # Far from center: reward = -|state| (worse)
        reward = -abs(self.state)
        
        # Episode ends if reached center or max steps reached
        done = (self.state == 0) or (self.steps >= self.max_steps)
        # Reached center: success
        # Max steps: timeout
        
        return self.state, reward, done  # Return (state, reward, done)

# ============================================
# REINFORCE AGENT: Policy Gradient Algorithm
# ============================================

# REINFORCEAgent: Implements REINFORCE (Monte Carlo Policy Gradient)
# Learns policy directly using gradient ascent on expected return
class REINFORCEAgent:
    def __init__(self, n_states, n_actions, learning_rate=0.01, gamma=0.99):
        """
        Initialize REINFORCE agent.
        
        Args:
            n_states: Number of possible states
            n_actions: Number of possible actions (2: left, right)
            learning_rate: Step size for policy parameter updates
            gamma: Discount factor for future rewards
        """
        self.n_states = n_states  # Store number of states
        self.n_actions = n_actions  # Store number of actions
        self.lr = learning_rate  # Learning rate: step size for gradient ascent
        self.gamma = gamma  # Discount factor: importance of future rewards
        
        # Policy parameters: θ (theta)
        # Shape: (n_states, n_actions)
        # For each state, we have parameters for each action
        # These parameters determine action probabilities via softmax
        self.theta = np.random.randn(n_states, n_actions) * 0.1
        # randn: sample from standard normal distribution
        # * 0.1: scale down (small initial values)
        # Small initialization: prevents extreme probabilities initially
    
    def policy(self, state):
        """
        Get action probabilities using softmax policy.
        
        Policy: π(a|s) = softmax(θ[s, a])
        Softmax converts logits (raw scores) to probabilities.
        
        Args:
            state: Current state
        
        Returns:
            Array of action probabilities [P(left), P(right)]
        """
        logits = self.theta[state]  # Get logits for this state
        # logits: raw scores for each action (before softmax)
        
        # Numerical stability: subtract max before exp
        # Prevents overflow when computing exp(logits)
        exp_logits = np.exp(logits - np.max(logits))
        # exp(logits - max): shift by max (doesn't change probabilities)
        
        # Normalize to get probabilities (sum to 1)
        probs = exp_logits / np.sum(exp_logits)
        # Softmax: exp(logit_i) / sum(exp(logit_j))
        # Result: probabilities that sum to 1
        
        return probs  # Return probability distribution over actions
    
    def select_action(self, state):
        """
        Sample action from policy (stochastic policy).
        
        Args:
            state: Current state
        
        Returns:
            Sampled action (0 or 1)
        """
        probs = self.policy(state)  # Get action probabilities
        # Sample action according to probability distribution
        return np.random.choice(self.n_actions, p=probs)
        # random.choice: sample from [0, 1] with probabilities probs
        # p=probs: use these probabilities
    
    def update(self, states, actions, rewards):
        """
        Update policy parameters using REINFORCE algorithm.
        
        REINFORCE update: θ ← θ + α * G_t * ∇_θ log π(a_t|s_t)
        
        Args:
            states: List of states in episode
            actions: List of actions taken
            rewards: List of rewards received
        """
        T = len(states)  # Episode length
        returns = []  # List to store discounted returns
        
        # Calculate discounted returns (backwards from end of episode)
        # Return G_t: sum of discounted rewards from time t to end
        G = 0  # Initialize return (starting from end)
        for t in reversed(range(T)):  # Go backwards through episode
            # G_t = r_t + γ * G_{t+1}
            G = rewards[t] + self.gamma * G
            # rewards[t]: immediate reward at time t
            # gamma * G: discounted future return
            # Insert at beginning (since we're going backwards)
            returns.insert(0, G)
            # returns[0] = G_T (return from last step)
            # returns[T-1] = G_0 (return from first step)
        
        # Update policy parameters for each step in episode
        for t in range(T):
            state = states[t]  # State at time t
            action = actions[t]  # Action taken at time t
            G_t = returns[t]  # Return from time t to end
            
            # Calculate policy gradient: ∇_θ log π(a_t|s_t)
            probs = self.policy(state)  # Current action probabilities
            
            # Gradient of log-probability
            # ∇_θ log π(a|s) = (1/π(a|s)) * ∇_θ π(a|s)
            grad_log_prob = np.zeros(self.n_actions)  # Initialize gradient
            # For the action taken: gradient is 1/probability
            grad_log_prob[action] = 1.0 / (probs[action] + 1e-8)
            # 1e-8: small epsilon to avoid division by zero
            # This is the gradient of log π(a_t|s_t) with respect to θ
            
            # REINFORCE update: θ ← θ + α * G_t * ∇_θ log π(a_t|s_t)
            # α (lr): learning rate
            # G_t: return (scalar)
            # grad_log_prob: gradient vector
            # probs: probabilities (for scaling)
            self.theta[state] += self.lr * G_t * grad_log_prob * probs
            # Update policy parameters for this state
            # Direction: increase probability of actions with high returns
            # Magnitude: proportional to return and learning rate

print("Environment and REINFORCE Agent classes defined!")  # Confirm classes are ready


In [ ]:
# ============================================
# TRAINING REINFORCE AGENT: Learning Optimal Policy
# ============================================

# Create environment: simple position control
env = SimpleEnv()  # Agent tries to stay at center (position 0)

# Create REINFORCE agent
agent = REINFORCEAgent(
    n_states=10,  # 10 possible states (positions -5 to +4, or similar)
    n_actions=2,  # 2 actions: left (0) or right (1)
    learning_rate=0.01,  # Learning rate: step size for policy updates
    gamma=0.99  # Discount factor: future rewards worth 99% of immediate
)

# Training parameters
num_episodes = 500  # Number of training episodes
rewards_per_episode = []  # Track total reward per episode
episode_lengths = []  # Track episode length (number of steps)

# Training loop: learn from experience
for episode in range(num_episodes):
    state = env.reset()  # Reset environment to initial state
    states = []  # Store states visited in this episode
    actions = []  # Store actions taken in this episode
    rewards = []  # Store rewards received in this episode
    done = False  # Whether episode finished
    
    # Generate episode: collect trajectory following current policy
    while not done:
        # Select action from current policy (stochastic)
        action = agent.select_action(state)  # Sample action from policy
        
        # Execute action in environment
        next_state, reward, done = env.step(action)
        # next_state: new position after action
        # reward: feedback from environment
        # done: whether episode ended
        
        # Store experience for this step
        states.append(state)  # Store state
        actions.append(action)  # Store action
        rewards.append(reward)  # Store reward
        
        # Move to next state
        state = next_state  # Update current state
    
    # Update policy using REINFORCE algorithm
    # Uses complete episode trajectory (Monte Carlo)
    agent.update(states, actions, rewards)
    # This is where learning happens: policy parameters get updated
    # Update happens after episode completes (not during)
    
    # Track performance metrics
    total_reward = sum(rewards)  # Sum all rewards in episode
    rewards_per_episode.append(total_reward)  # Store total reward
    episode_lengths.append(len(states))  # Store episode length
    
    # Print progress every 100 episodes
    if (episode + 1) % 100 == 0:
        # Calculate average performance over last 100 episodes
        avg_reward = np.mean(rewards_per_episode[-100:])  # Average reward
        avg_length = np.mean(episode_lengths[-100:])  # Average length
        # Print statistics
        print(f"Episode {episode+1}: Avg Reward = {avg_reward:.2f}, "
              f"Avg Length = {avg_length:.2f}")
        # avg_reward: should increase over time (less negative, closer to 0)
        # avg_length: may vary depending on task

# Training complete: print final statistics
print(f"\nTraining complete!")
print(f"Average reward (last 100 episodes): {np.mean(rewards_per_episode[-100:]):.2f}")
# Higher average reward = better learned policy
# For this task: reward closer to 0 is better (stays at center)


## Learning Progress

Let's visualize the learning progress.


In [ ]:
# ============================================
# VISUALIZING LEARNING PROGRESS: Tracking Performance
# ============================================

# Create figure with two subplots side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# fig: figure object (entire plot)
# axes: array of axis objects (one for each subplot)
# 1, 2: 1 row, 2 columns
# figsize: width=14 inches, height=5 inches

# ============================================
# PLOT 1: Reward per Episode
# ============================================

# Plot raw reward data (thin, semi-transparent line)
axes[0].plot(rewards_per_episode, alpha=0.6, linewidth=0.5)
# alpha=0.6: 60% opacity (semi-transparent)
# linewidth=0.5: thin line
# Shows individual episode rewards (noisy, high variance)

# Calculate and plot moving average (smoother trend)
window = 50  # Number of episodes to average over
if len(rewards_per_episode) >= window:
    # Calculate moving average using convolution
    # np.ones(window)/window: array of 1/window (for averaging)
    # np.convolve: sliding window average
    # mode='valid': only compute where window fully overlaps
    moving_avg = np.convolve(rewards_per_episode, np.ones(window)/window, mode='valid')
    # Result: smoother line showing overall trend
    
    # Plot moving average (thick, red line)
    # range(window-1, len(...)): x-axis indices for moving average
    axes[0].plot(range(window-1, len(rewards_per_episode)), moving_avg, 
                color='red', linewidth=2, label=f'Moving Average ({window})')
    # color='red': red line for visibility
    # linewidth=2: thick line
    # label: for legend

# Add labels and formatting
axes[0].set_xlabel('Episode')  # X-axis: episode number
axes[0].set_ylabel('Total Reward')  # Y-axis: cumulative reward
axes[0].set_title('Reward per Episode')  # Plot title
axes[0].legend()  # Show legend (moving average label)
axes[0].grid(True, alpha=0.3)  # Add grid (30% opacity)
# Grid helps read values from plot

# ============================================
# PLOT 2: Episode Length
# ============================================

# Plot raw episode length data (thin, green, semi-transparent line)
axes[1].plot(episode_lengths, alpha=0.6, linewidth=0.5, color='green')
# color='green': different color to distinguish from rewards
# Shows individual episode lengths (noisy)

# Calculate and plot moving average
if len(episode_lengths) >= window:
    # Same as above: calculate moving average
    moving_avg_len = np.convolve(episode_lengths, np.ones(window)/window, mode='valid')
    # Smooth out noise
    
    # Plot moving average (thick, red line)
    axes[1].plot(range(window-1, len(episode_lengths)), moving_avg_len, 
                color='red', linewidth=2, label=f'Moving Average ({window})')

# Add labels and formatting
axes[1].set_xlabel('Episode')  # X-axis: episode number
axes[1].set_ylabel('Episode Length')  # Y-axis: number of steps
axes[1].set_title('Episode Length')  # Plot title
axes[1].legend()  # Show legend
axes[1].grid(True, alpha=0.3)  # Add grid

# Adjust layout to prevent overlap
plt.tight_layout()  # Automatically adjust spacing
plt.show()  # Display the plots

# ============================================
# INTERPRETATION
# ============================================

# Expected patterns:
# 1. Rewards: Should increase over time (less negative, closer to 0)
#    - Early episodes: very negative (agent far from center)
#    - Later episodes: less negative (agent learns to stay near center)
#    - High variance: policy gradients have high variance (normal)
# 2. Episode lengths: May vary depending on task
#    - Could increase if agent explores more
#    - Could decrease if agent finds efficient strategy
# 3. Moving averages: Should show clear upward (rewards) trend
#    - Smoothes out high variance in policy gradients


## Policy Visualization

Let's visualize the learned policy.


In [ ]:
# ============================================
# VISUALIZING LEARNED POLICY: Action Probabilities
# ============================================

# Extract action probabilities for all states
states_range = range(agent.n_states)  # All possible states
# Calculate policy (action probabilities) for each state
action_probs = np.array([agent.policy(s) for s in states_range])
# action_probs: shape (n_states, n_actions)
# action_probs[i, 0] = P(left | state i)
# action_probs[i, 1] = P(right | state i)

# Create bar chart showing action probabilities
plt.figure(figsize=(10, 6))  # 10x6 inch figure
x = np.arange(len(states_range))  # X-axis positions (state indices)
width = 0.35  # Width of bars

# Plot bars for each action
# Action 0 (Left): bars shifted left
plt.bar(x - width/2, action_probs[:, 0], width, 
        label='Action 0 (Left)', alpha=0.7)
# x - width/2: shift bars left
# action_probs[:, 0]: probabilities of left action for all states
# width: bar width
# label: for legend
# alpha=0.7: 70% opacity

# Action 1 (Right): bars shifted right
plt.bar(x + width/2, action_probs[:, 1], width, 
        label='Action 1 (Right)', alpha=0.7)
# x + width/2: shift bars right
# action_probs[:, 1]: probabilities of right action for all states

# Add labels and formatting
plt.xlabel('State')  # X-axis label
plt.ylabel('Probability')  # Y-axis label
plt.title('Learned Policy (Action Probabilities)')  # Plot title
plt.xticks(x, states_range)  # X-axis ticks: state numbers
plt.legend()  # Show legend (action labels)
plt.grid(True, alpha=0.3, axis='y')  # Add horizontal grid lines
plt.tight_layout()  # Adjust spacing
plt.show()  # Display plot

# ============================================
# PRINT POLICY DETAILS: Numerical Probabilities
# ============================================

# Show policy probabilities for first few states
print("Learned Policy:")
for state in range(min(5, agent.n_states)):  # Show first 5 states
    probs = agent.policy(state)  # Get action probabilities for this state
    # Print probabilities for both actions
    print(f"  State {state}: P(Left)={probs[0]:.3f}, P(Right)={probs[1]:.3f}")
    # probs[0]: probability of left action
    # probs[1]: probability of right action
    # .3f: format to 3 decimal places

# ============================================
# INTERPRETATION
# ============================================

# Policy visualization shows:
# - Which action is preferred in each state
# - How confident the policy is (probabilities close to 0 or 1 = confident)
# - Stochastic vs deterministic: probabilities near 0.5 = stochastic, near 0/1 = deterministic
# 
# For this task (staying at center):
# - States far from center: should prefer action that moves toward center
# - State at center: probabilities may be balanced (both actions keep at center)
# - Learned policy should show clear preference patterns


## Validation & Testing

Let's test the learned policy and compare with value-based methods.


In [ ]:
# ============================================
# TESTING LEARNED POLICY: Evaluate Performance
# ============================================

# Test learned policy on fresh episodes
test_episodes = 10  # Number of test episodes
test_rewards = []  # Store test rewards

# Run test episodes
for episode in range(test_episodes):
    env_test = SimpleEnv()  # Create fresh environment
    state = env_test.reset()  # Reset to initial state
    total_reward = 0  # Accumulate reward
    done = False  # Episode finished flag
    
    # Run episode until done
    while not done:
        # Select action from learned policy (stochastic)
        action = agent.select_action(state)  # Sample from policy
        next_state, reward, done = env_test.step(action)  # Execute action
        state = next_state  # Move to next state
        total_reward += reward  # Accumulate reward
    
    # Record test result
    test_rewards.append(total_reward)  # Store total reward

# Print test results
print("Test Results:")
print(f"  Average reward: {np.mean(test_rewards):.2f}")  # Mean reward
print(f"  Std reward: {np.std(test_rewards):.2f}")  # Standard deviation
# Higher mean = better performance
# Lower std = more consistent performance

# ============================================
# HYPERPARAMETER COMPARISON: Learning Rate Effect
# ============================================

# Compare different learning rates to see their effect
learning_rates = [0.001, 0.01, 0.1]  # Different learning rates to test
# 0.001: very slow learning (small updates)
# 0.01: moderate learning (default)
# 0.1: fast learning (large updates)

lr_results = []  # Store results for each learning rate

# Test each learning rate
for lr in learning_rates:
    # Create new environment and agent with this learning rate
    env_lr = SimpleEnv()
    agent_lr = REINFORCEAgent(
        n_states=10, n_actions=2,
        learning_rate=lr,  # Use this learning rate
        gamma=0.99  # Same discount factor
    )
    
    # Train for fewer episodes (for faster comparison)
    for episode in range(200):  # 200 episodes (less than full training)
        state = env_lr.reset()  # Reset environment
        states, actions, rewards = [], [], []  # Store episode data
        done = False
        
        # Generate episode
        while not done:
            action = agent_lr.select_action(state)  # Choose action
            next_state, reward, done = env_lr.step(action)  # Execute
            # Store experience
            states.append(state)  # Store state
            actions.append(action)  # Store action
            rewards.append(reward)  # Store reward
            state = next_state  # Move to next state
        
        # Update policy
        agent_lr.update(states, actions, rewards)  # Learn from episode
    
    # Test learned policy
    test_reward = 0  # Accumulate reward
    state = env_lr.reset()  # Reset environment
    done = False
    while not done:
        action = agent_lr.select_action(state)  # Sample action
        next_state, reward, done = env_lr.step(action)  # Execute
        state = next_state  # Move
        test_reward += reward  # Accumulate
    
    # Store result
    lr_results.append({'lr': lr, 'reward': test_reward})
    print(f"  Learning rate {lr}: Test reward = {test_reward:.2f}")

# ============================================
# VALIDATION: Check Learning Success
# ============================================

# Assertion: verify agent learned reasonable policy
assert np.mean(test_rewards) > -50, "Agent should learn reasonable policy"
# Mean reward should be reasonable (not too negative)
# -50 is a threshold: agent should perform better than random
print("\n✓ Validation checks passed")
# If assertion passes, agent learned successfully


## Summary & Key Takeaways

### Key Concepts Learned

1. **Policy Gradient Basics**
   - Direct policy optimization
   - Uses gradient ascent on expected return
   - Policy gradient theorem provides gradient
   - REINFORCE: Monte Carlo policy gradient

2. **REINFORCE Algorithm**
   - Generate episode following current policy
   - Calculate returns (discounted sum of rewards)
   - Update policy using gradient of log-probability
   - High variance but unbiased

3. **Policy vs Value Methods**
   - **Policy-based**: Direct policy optimization
   - **Value-based**: Learn value function, derive policy
   - **Actor-Critic**: Combines both (reduces variance)

4. **Key Hyperparameters**
   - **learning_rate**: Step size for policy updates
   - **gamma**: Discount factor
   - **baseline**: Variance reduction (optional)

### When to Use Policy Gradient Methods

✅ **Good for:**
- Continuous action spaces
- High-dimensional state spaces
- Stochastic policies needed
- Complex control problems
- When value function is hard to estimate
- Robotics and continuous control

❌ **Not ideal for:**
- Simple discrete problems (Q-learning better)
- When sample efficiency is critical
- Very large action spaces
- When deterministic policy is sufficient
- Real-time applications (slow)

### Next Steps

- Try **Actor-Critic** methods (reduce variance)
- Explore **PPO (Proximal Policy Optimization)** for stability
- Use **A3C** for parallel training
- Apply to **gym continuous control** environments
- Experiment with **baseline** for variance reduction
